In [33]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import re

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

In [34]:
CFG = {
    "model_name": "convnextv2_tiny.fcmae_ft_in22k_in1k",
    "img_size": 384,
    "batch_size": 32,
    "num_classes": 3,
    "num_workers": 0,
    "device": "cuda"
}

DATA_DIR = Path("../data")
TEST_DIR = DATA_DIR / "test"

WEIGHT_DIR = Path("../weights")

In [35]:
original_transform = A.Compose([
    A.LongestMaxSize(max_size=CFG["img_size"]),
    A.PadIfNeeded(
        min_height=CFG["img_size"],
        min_width=CFG["img_size"],
        border_mode=cv2.BORDER_CONSTANT
    ),
    A.Normalize(),
    ToTensorV2()
])

horizontal_flip_transform = A.Compose([
    A.LongestMaxSize(max_size=CFG["img_size"]),
    A.PadIfNeeded(
        min_height=CFG["img_size"],
        min_width=CFG["img_size"],
        border_mode=cv2.BORDER_CONSTANT
    ),
    A.HorizontalFlip(p=1),
    A.Normalize(),
    ToTensorV2()
])

tta_transforms = [
    original_transform,
    horizontal_flip_transform
]

In [36]:
def natural_sort_key(path):
    return [int(s) if s.isdigit() else s.lower()
            for s in re.split(r'(\d+)', path.name)]

test_files = list(TEST_DIR.glob("*"))
test_files.sort(key=natural_sort_key)

test_df = pd.DataFrame({
    "filepath": test_files
})

test_df["id"] = test_df["filepath"].apply(lambda x: x.name)

print(test_df.shape)
test_df.head()

(1458, 2)


,filepath,id
0,..\data\test\1.jpg,1.jpg
1,..\data\test\2.jpg,2.jpg
2,..\data\test\3.jpg,3.jpg
3,..\data\test\4.jpg,4.jpg
4,..\data\test\5.jpg,5.jpg


In [37]:
class TestDataset(Dataset):

    def __init__(self, df, transforms=None):

        self.df = df
        self.transforms = transforms

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        img_path = self.df.iloc[idx]["filepath"]

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            image = self.transforms(image=image)["image"]

        return image

In [38]:
def load_model(weight_path):

    model = timm.create_model(
        CFG["model_name"],
        pretrained=False,
        num_classes=CFG["num_classes"]
    )

    model.load_state_dict(
        torch.load(weight_path, map_location=CFG["device"])
    )

    model.to(CFG["device"])
    model.eval()

    return model

In [39]:
models = []

for fold in range(5):

    weight_path = WEIGHT_DIR / f"best_fold{fold}.pth"

    model = load_model(weight_path)

    models.append(model)

print(f"Loaded {len(models)} models.")

Loaded 5 models.


In [40]:
tta_probs = []

for i, transform in enumerate(tta_transforms):
    print(f"TTA {i+1}/{len(tta_transforms)}")

    test_dataset = TestDataset(
        test_df,
        transform
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=CFG["batch_size"],
        shuffle=False,
        num_workers=CFG["num_workers"],
        pin_memory=True
    )

    probs_all = []

    with torch.no_grad():

        for images in tqdm(test_loader):

            images = images.to(CFG["device"])

            probs = torch.zeros(
                (images.size(0), CFG["num_classes"]),
                device=CFG["device"]
            )

            for model in models:

                outputs = model(images)

                probs += torch.softmax(outputs, dim=1)

            probs /= len(models)

            probs_all.append(probs.cpu().float().numpy())

    tta_probs.append(np.concatenate(probs_all))

TTA 1/2


100%|██████████| 46/46 [01:14<00:00,  1.62s/it]


TTA 2/2


100%|██████████| 46/46 [01:04<00:00,  1.40s/it]


In [41]:
tta_probs = np.mean(tta_probs, axis=0)

predictions = np.argmax(tta_probs, axis=1)

In [42]:
sample_submission = pd.read_csv("../data/submission.csv")

sample_submission["predicted"] = predictions

sample_submission.to_csv(
    "../output/submission_5fold_convnext_tta.csv",
    index=False
)

sample_submission.head()

,id,predicted
0,1,2
1,2,2
2,3,2
3,4,1
4,5,0


In [43]:
print(len(predictions))
print(len(sample_submission))

1458
1458


In [44]:
print(sample_submission["predicted"].value_counts())

predicted
2    736
0    520
1    202
Name: count, dtype: int64
